# Gray–Scott dataset generation

Generate the ten initial-condition trajectories used by SLD. The equations, RK4 integrator, seeds, parameters, simulation length, and sampling logic are preserved from the supplied notebook.

Each IC produces an HR MAT file `2DGS_IC{ic}_ri10_2x3001x256x256.mat` and a clean LR MAT file `2DGS_IC{ic}_ri10_2x751x32x32.mat`, both with key `uv`. The retained primary cell also saves the original zero-noise (`obsn00`) LR variant. Training uses the clean LR filenames.

Data are saved under `<repository>/data/gray_scott`; generated figures are under `<repository>/results/data_generation/gray_scott/figures`. Override those roots with `SLD_DATA_ROOT` and `SLD_RESULTS_ROOT`. Install `pyrtools` to retain the binomial anti-aliased downsampling used in the paper.

The original float64 HR array is about 3.15 GB per IC (31.5 GB for ten ICs, before LR files). A single trajectory uses several large arrays and repeated concatenation, so allow ample RAM, disk space, and runtime. The full simulation has not been executed as part of repository preparation. Optional cells generate noisy variants or display existing dataset plots.


In [ ]:
from pathlib import Path
import os

# Locate the repository from the notebook folder or repository root.
_start = Path.cwd().resolve()
ROOT = next((p for p in (_start, *_start.parents)
             if (p / "README.md").is_file() and (p / "data_generation").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the SLD repository or one of its subfolders.")
DATA_ROOT = Path(os.environ.get("SLD_DATA_ROOT", str(ROOT / "data"))).expanduser().resolve()
RESULTS_ROOT = Path(os.environ.get("SLD_RESULTS_ROOT", str(ROOT / "results"))).expanduser().resolve()


## Generate the clean dataset

Running this cell simulates all ten trajectories. It also keeps the original zero-noise LR export.


In [ ]:
import numpy as np
import scipy.io as scio
from pathlib import Path
import torch
import torch.nn as nn
import numpy as np
import scipy.io as scio
import os
import matplotlib.pyplot as plt
from pathlib import Path
try:
    import pyrtools as pt
    HAS_PYRTOOLS = True
except ImportError as exc:
    raise ImportError("Install pyrtools from requirements.txt for the paper's anti-aliased downsampling.") from exc

# Create directories
data_dir = DATA_ROOT / 'gray_scott'
fig_dir = RESULTS_ROOT / 'data_generation' / 'gray_scott' / 'figures'
data_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

torch.manual_seed(1) 
np.random.seed(1) 

lapl_op = [[[[    0,   0, -1/12,   0,     0],
             [    0,   0,   4/3,   0,     0],
             [-1/12, 4/3,    -5, 4/3, -1/12],
             [    0,   0,   4/3,   0,     0],
             [    0,   0, -1/12,   0,     0]]]]

def apply_laplacian(mat, dx = 0.01):
    """This function applies a discretized Laplacian
    in periodic boundary conditions to a matrix"""
    
    neigh_mat = -5*mat.copy()
    
    neighbors = [
                    ( 4/3,  (-1, 0) ),
                    ( 4/3,  ( 0,-1) ),
                    ( 4/3,  ( 0, 1) ),
                    ( 4/3,  ( 1, 0) ),
                    (-1/12,  (-2, 0)),
                    (-1/12,  (0, -2)),
                    (-1/12,  (0, 2)),
                    (-1/12,  (2, 0)),
                ]
    
    for weight, neigh in neighbors:
        neigh_mat += weight * np.roll(mat, neigh, (0,1))
    
    return neigh_mat/dx**2

def update_rk4(A0, B0, DA, DB, f, k, delta_t, dx):
    """Update with Runge-kutta-4 method"""
    ############# Stage 1 ##############
    diff_A = DA * apply_laplacian(A0, dx)
    diff_B = DB * apply_laplacian(B0, dx)
    
    reaction = A0 * B0 ** 2
    diff_A -= reaction
    diff_B += reaction
    
    diff_A += f * (1 - A0)
    diff_B -= (k + f) * B0
    
    K1_a = diff_A
    K1_b = diff_B
    
    ############# Stage 2 ##############
    A1 = A0 +  K1_a * delta_t/2.0
    B1 = B0 +  K1_b * delta_t/2.0
    
    diff_A = DA * apply_laplacian(A1, dx)
    diff_B = DB * apply_laplacian(B1, dx)
    
    reaction = A1 * B1 ** 2
    diff_A -= reaction
    diff_B += reaction
    
    diff_A += f * (1 - A1)
    diff_B -= (k + f) * B1
    
    K2_a = diff_A
    K2_b = diff_B
    
    ############# Stage 3 ##############
    A2 = A0 + K2_a * delta_t/2.0
    B2 = B0 + K2_b * delta_t/2.0
    
    diff_A = DA * apply_laplacian(A2, dx)
    diff_B = DB * apply_laplacian(B2, dx)
    
    reaction = A2 * B2 ** 2
    diff_A -= reaction
    diff_B += reaction
    
    diff_A += f * (1 - A2)
    diff_B -= (k + f) * B2
    
    K3_a = diff_A
    K3_b = diff_B
    
    ############# Stage 4 ##############
    A3 = A0 + K3_a * delta_t
    B3 = B0 + K3_b * delta_t
    
    diff_A = DA * apply_laplacian(A3, dx)
    diff_B = DB * apply_laplacian(B3, dx)
    
    reaction = A3 * B3 ** 2
    diff_A -= reaction
    diff_B += reaction
    
    diff_A += f * (1 - A3)
    diff_B -= (k + f) * B3
    
    K4_a = diff_A
    K4_b = diff_B
    
    # Final solution
    A = A0 + delta_t*(K1_a+2*K2_a+2*K3_a+K4_a)/6.0
    B = B0 + delta_t*(K1_b+2*K2_b+2*K3_b+K4_b)/6.0
    
    return A, B

def postProcess(output, N, xmin, xmax, ymin, ymax, num, batch, save_dir):
    """Visualization function
    Args:
        output: array with shape [2, T, H, W] or [T, 2, H, W]
        num: time index to visualize
    """
    x = np.linspace(xmin, xmax, N+1)[:-1]
    y = np.linspace(ymin, ymax, N+1)[:-1]
    x_star, y_star = np.meshgrid(x, y)
    
    # Handle both possible shapes
    if output.shape[0] == 2:  # Shape is [2, T, H, W]
        u_pred = output[0, num, :, :]
        v_pred = output[1, num, :, :]
    else:  # Shape is [T, 2, H, W]
        u_pred = output[num, 0, :, :]
        v_pred = output[num, 1, :, :]
    
    fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(6, 3))
    fig.subplots_adjust(hspace=0.3, wspace=0.3)
    
    cf = ax[0].scatter(x_star, y_star, c=u_pred, alpha=0.95, edgecolors='none', 
                       cmap='hot', marker='s', s=2)
    ax[0].axis('square')
    ax[0].set_xlim([xmin, xmax])
    ax[0].set_ylim([ymin, ymax])
    ax[0].set_title('u-FDM')
    fig.colorbar(cf, ax=ax[0])
    
    cf = ax[1].scatter(x_star, y_star, c=v_pred, alpha=0.95, edgecolors='none', 
                       cmap='hot', marker='s', s=2)
    ax[1].axis('square')
    ax[1].set_xlim([xmin, xmax])
    ax[1].set_ylim([ymin, ymax])
    ax[1].set_title('v-FDM')
    fig.colorbar(cf, ax=ax[1])
    
    plt.savefig(save_dir / f'uv_[b={batch}][t={num}].png', dpi=150, bbox_inches='tight')
    plt.close('all')

def get_initial_A_and_B(N, random_influence=0.2, ic_seed=1):
    """Get initial chemical concentrations
    完全对齐论文原始方式：
    - 背景噪声由 ic_seed 控制（每个IC不同）
    - 中心块固定在正中间（不偏移）
    """
    np.random.seed(ic_seed)  # ← 在函数内部设种子，和原始代码一致

    A = (1 - random_influence) * np.ones((N, N))
    B = np.zeros((N, N))
    A += random_influence * np.random.random((N, N))
    B += random_influence * np.random.random((N, N))

    # 中心块固定正中，不加偏移
    N2 = N // 2
    r  = int(N / 10.0)
    A[N2-r:N2+r, N2-r:N2+r] = 0.50
    B[N2-r:N2+r, N2-r:N2+r] = 0.25

    return A, B


def downsample_spatial_official(data_high, n_levels=3):
    """Downsample spatially using pyrtools (official method)
    
    Args:
        data_high: [2, T, 256, 256] array
        n_levels: pyramid levels (3 gives 256/8 = 32)
    
    Returns:
        data_low: [2, T, 32, 32] array
    """
    if not HAS_PYRTOOLS:
        print("Using simple decimation (not official method!)")
        return data_high[:, :, ::8, ::8]
    
    T = data_high.shape[1]
    data_low = np.zeros([2, T, 32, 32])
    
    for t in range(T):
        data_low[0, t, :, :] = pt.blurDn(data_high[0, t, :, :], n_levels, filt='binom5')
        data_low[1, t, :, :] = pt.blurDn(data_high[1, t, :, :], n_levels, filt='binom5')
    
    return data_low

def downsample_temporal(data, t_ds=4):
    """Downsample temporally by taking every t_ds-th timestep
    
    Args:
        data: [2, T, H, W] array
        t_ds: temporal downsampling factor
    
    Returns:
        downsampled: [2, T//t_ds, H, W] array
    """
    return data[:, ::t_ds, :, :]


def add_observation_noise_lr(UV_low_clean, noise_level=0.10, noise_type="gaussian",
                             seed=None, per_channel=True, clip=None):
    """
    Add observation noise to LR only.

    Args:
        UV_low_clean: [2, T, 32, 32] clean LR
        noise_level:  e.g., 0.10 means sigma = 0.10 * std(LR)  (per-channel if per_channel=True)
        noise_type:   "gaussian" (recommended)
        seed:         int or None (for reproducibility)
        per_channel:  if True, scale sigma by each channel's std separately
        clip:         tuple(min, max) or None. If not None, clip noisy LR.

    Returns:
        UV_low_noisy: same shape as UV_low_clean
    """
    rng = np.random.default_rng(seed)
    UV = UV_low_clean.astype(np.float32, copy=True)

    if noise_type.lower() != "gaussian":
        raise ValueError(f"Unsupported noise_type: {noise_type}")

    if per_channel:
        # std over (T,H,W) for each channel
        ch_std = UV.reshape(2, -1).std(axis=1)  # shape (2,)
        sigma = (noise_level * ch_std).reshape(2, 1, 1, 1)
    else:
        sigma_scalar = noise_level * UV.std()
        sigma = sigma_scalar

    noise = rng.normal(loc=0.0, scale=1.0, size=UV.shape).astype(np.float32)
    UV_noisy = UV + noise * sigma

    if clip is not None:
        UV_noisy = np.clip(UV_noisy, clip[0], clip[1])

    return UV_noisy


def generate_dataset(ic_number, N=256, N_simulation_steps=15000,
                     random_influence=0.1,
                     ic_style='official', use_official_downsample=True,
                     add_obs_noise=False, obs_noise_level=0.10, obs_noise_seed_base=12345,
                     obs_noise_per_channel=True, save_noisy_lr=True):

    """Generate a single dataset with given initial condition"""
    
    # Model parameters (from official code)
    DA = 0.16
    DB = 0.08
    f = 0.06
    k = 0.062
    delta_t = 1.0
    dx = 1.0
    
    # Downsampling parameters (from official downsampling script)
    TEMPORAL_DS = 4  # Temporal downsampling factor
    SPATIAL_DS = 8   # Spatial downsampling factor (2^3 = 8)
    N_LEVELS = 3     # Pyramid levels

    # ✅ [修改1] 把 random_influence 编码进文件名tag，避免覆盖旧数据
    ri_tag = f"ri{int(random_influence * 100):02d}"  # 0.1 → "ri10", 0.2 → "ri20"
    
    print(f"\n{'='*60}")
    print(f"Generating dataset for IC={ic_number}, random_influence={random_influence} (tag: {ri_tag})")
    print(f"{'='*60}")
    print(f"Initial condition style: {ic_style}")
    print(f"Temporal downsampling: {TEMPORAL_DS}x")
    print(f"Spatial downsampling: {SPATIAL_DS}x (pyramid levels: {N_LEVELS})")
    
    # ✅ [修改2] 传入 random_influence 和 ic_seed，让每个IC真正不同
    A, B = get_initial_A_and_B(N, random_influence=random_influence, ic_seed=ic_number)
    A_record = A.copy()[None,...]
    B_record = B.copy()[None,...]
    
    # Simulate
    for step in range(N_simulation_steps):
        A, B = update_rk4(A, B, DA, DB, f, k, delta_t, dx)
        
        if step % 5 == 0:
            if step % 1000 == 0:
                print(f"  Progress: {step}/{N_simulation_steps}")
            A_record = np.concatenate((A_record, A[None,...]), axis=0)
            B_record = np.concatenate((B_record, B[None,...]), axis=0)
    
    print(f"  Simulation complete: {A_record.shape[0]} timesteps")
    
    # Save high-resolution data
    UV_high = np.stack([A_record, B_record], axis=0)  # [2, 3001, 256, 256]
    # ✅ [修改3] 文件名加入 ri_tag
    save_path_high = data_dir / f'2DGS_IC{ic_number}_{ri_tag}_2x3001x256x256.mat'
    scio.savemat(save_path_high, {'uv': UV_high})
    print(f"  ✓ Saved HR: {save_path_high.name} | shape: {UV_high.shape}")
    
    # Generate low-resolution data (matching official method)
    if use_official_downsample:
        # Step 1: Temporal downsampling (every 4th timestep)
        UV_temp = downsample_temporal(UV_high, t_ds=TEMPORAL_DS)
        print(f"  After temporal DS: {UV_temp.shape}")
        
        # Step 2: Spatial downsampling (pyrtools blur + decimate)
        UV_low = downsample_spatial_official(UV_temp, n_levels=N_LEVELS)
        print(f"  After spatial DS: {UV_low.shape}")
    else:
        # Simple decimation (not recommended!)
        UV_low = UV_high[:, ::TEMPORAL_DS, ::SPATIAL_DS, ::SPATIAL_DS]
        print(f"  Warning: Using simple decimation (not official method)")
    
    # Save low-resolution data
    # ✅ [修改4] 文件名加入 ri_tag
    # Save low-resolution data (clean)
    save_path_low = data_dir / f'2DGS_IC{ic_number}_{ri_tag}_2x751x32x32.mat'
    scio.savemat(save_path_low, {'uv': UV_low})
    print(f"  ✓ Saved LR: {save_path_low.name} | shape: {UV_low.shape}")
    
    # -------------------------------
    # Scheme B: add observation noise on LR (do NOT touch HR)
    # -------------------------------
    if add_obs_noise and save_noisy_lr:
        # Tag for observation noise (e.g., 0.10 -> "obsn10")
        obsn_tag = f"obsn{int(obs_noise_level * 100):02d}"

        # Make it reproducible per-IC (and independent from IC init seed)
        noise_seed = int(obs_noise_seed_base + ic_number)

        UV_low_noisy = add_observation_noise_lr(
            UV_low_clean=UV_low,
            noise_level=obs_noise_level,
            noise_type="gaussian",
            seed=noise_seed,
            per_channel=obs_noise_per_channel,
            clip=None  # or set (0.0, 1.0) if you want
        )

        # IMPORTANT: do not overwrite the clean LR file
        save_path_low_noisy = data_dir / f'2DGS_IC{ic_number}_{ri_tag}_{obsn_tag}_2x751x32x32.mat'
        scio.savemat(save_path_low_noisy, {'uv': UV_low_noisy})

        print(f"  ✓ Saved LR (noisy): {save_path_low_noisy.name} | shape: {UV_low_noisy.shape} | seed: {noise_seed}")


    
    # Verification
    assert UV_high.shape == (2, 3001, 256, 256), f"HR shape mismatch: {UV_high.shape}"
    assert UV_low.shape == (2, 751, 32, 32), f"LR shape mismatch: {UV_low.shape}"
    print(f"  ✓ Shape verification passed")
    
    # Optional: visualize (only for IC=1)
    if ic_number == 1:
        print("  Generating visualizations...")
        for t_idx in [0, 500, 1000, 1500, 2000, 2500]:
            if t_idx < UV_high.shape[1]:
                postProcess(UV_high, N, 0, N, 0, N, t_idx, ic_number, fig_dir)
    
    return UV_high, UV_low


# Main execution
if __name__ == '__main__':
    print("="*60)
    print("2D Gray-Scott Dataset Generation")
    print("="*60)
    print(f"Data directory: {data_dir.absolute()}")
    print(f"Figure directory: {fig_dir.absolute()}")

    if not HAS_PYRTOOLS:
        print("\n⚠️  WARNING: pyrtools not installed!")
        print("Install with: pip install pyrtools")
        print("Without it, downsampling won't match official method!\n")

    # ✅ 统一在这里设置 random_influence，所有IC保持一致
    RANDOM_INFLUENCE = 0.1
    OBS_NOISE_LEVEL  = 0

    for ic in range(1, 11):
        generate_dataset(
            ic_number=ic,
            N=256,
            N_simulation_steps=15000,
            random_influence=RANDOM_INFLUENCE,
            ic_style='official',
            use_official_downsample=HAS_PYRTOOLS,
            add_obs_noise=True,
            obs_noise_level=OBS_NOISE_LEVEL,
            obs_noise_seed_base=20250301
        )

    print("\n" + "="*60)
    print("Dataset Generation Complete!")
    print(f"Created {len(list(data_dir.glob('*_2x3001x256x256.mat')))} HR files")
    print(f"Created {len(list(data_dir.glob('*_2x751x32x32.mat')))} LR files")
    print("="*60)


## Optional observation-noise variants

Requires the paired dataset files from the generation cell. Enable the flag to run.


In [ ]:
GENERATE_NOISY_VARIANTS = False
if GENERATE_NOISY_VARIANTS:
    import numpy as np
    import scipy.io as scio
    from pathlib import Path

    data_dir = DATA_ROOT / 'gray_scott'

    def add_observation_noise_lr(UV_low_clean, noise_level=0.10, seed=None,
                                 per_channel=True, clip=None):
        """
        UV_low_clean: [2, T, H, W]
        noise_level: 0.10 means sigma = 0.10 * std(LR)
        """
        rng = np.random.default_rng(seed)
        UV = UV_low_clean.astype(np.float32, copy=True)

        if per_channel:
            ch_std = UV.reshape(UV.shape[0], -1).std(axis=1)
            sigma = (noise_level * ch_std).reshape(UV.shape[0], 1, 1, 1)
        else:
            sigma = noise_level * UV.std()

        noise = rng.normal(0.0, 1.0, size=UV.shape).astype(np.float32)
        UV_noisy = UV + noise * sigma

        if clip is not None:
            UV_noisy = np.clip(UV_noisy, clip[0], clip[1])

        return UV_noisy


    noise_levels = [0.01, 0.05, 0.10, 0.20]
    seed_base = 20250301

    for noise_level in noise_levels:
        obsn_tag = f"obsn{int(noise_level * 100):02d}"

        # 只匹配 clean LR，避免把 noisy LR 再加噪
        lr_files = sorted(data_dir.glob("2DGS_IC*_ri10_2x751x32x32.mat"))

        for lr_path in lr_files:
            name = lr_path.name

            # 跳过已经是 noisy 的文件
            if "obsn" in name:
                continue

            data = scio.loadmat(lr_path)
            UV_low = data["uv"].astype(np.float32)

            # 从文件名中解析 IC 编号
            # e.g. 2DGS_IC3_ri10_2x751x32x32.mat
            ic_str = name.split("_")[1]  # IC3
            ic_number = int(ic_str.replace("IC", ""))

            seed = seed_base + ic_number + int(noise_level * 1000)

            UV_noisy = add_observation_noise_lr(
                UV_low_clean=UV_low,
                noise_level=noise_level,
                seed=seed,
                per_channel=True,
                clip=None,
            )

            save_name = name.replace("_2x751x32x32.mat", f"_{obsn_tag}_2x751x32x32.mat")
            save_path = data_dir / save_name

            scio.savemat(save_path, {"uv": UV_noisy})
            print(f"Saved {save_path.name} | noise={noise_level} | seed={seed}")


## Optional dataset overview plots

Requires the paired dataset files from the generation cell. Enable the flag to run.


In [ ]:
RUN_DATA_OVERVIEW_PLOTS = False
if RUN_DATA_OVERVIEW_PLOTS:
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    import scipy.io as scio
    import numpy as np

    def visualize_dataset(data_dir, ic_number=1, random_influence=0.1):
        """
        可视化单个IC的HR和LR数据
        - 展示 U 和 V 两个通道
        - 展示多个时间步
        """
        ri_tag = f"ri{int(random_influence * 100):02d}"
    
        # 加载数据
        hr_path = data_dir / f'2DGS_IC{ic_number}_{ri_tag}_2x3001x256x256.mat'
        lr_path = data_dir / f'2DGS_IC{ic_number}_{ri_tag}_2x751x32x32.mat'
    
        print(f"Loading IC={ic_number}, tag={ri_tag}...")
        hr = scio.loadmat(hr_path)['uv']  # [2, 3001, 256, 256]
        lr = scio.loadmat(lr_path)['uv']  # [2, 751, 32, 32]
        print(f"  HR shape: {hr.shape}")
        print(f"  LR shape: {lr.shape}")

        # ── 图1：多时间步快照（U通道）──────────────────────────────
        t_indices_hr = [0, 500, 1000, 1500, 2000, 2500, 3000]
        fig, axes = plt.subplots(2, len(t_indices_hr), figsize=(20, 6))
        fig.suptitle(f'IC={ic_number} | HR U channel (256×256) | ri={random_influence}', fontsize=13)

        for col, t in enumerate(t_indices_hr):
            if t < hr.shape[1]:
                im = axes[0, col].imshow(hr[0, t], cmap='RdBu_r', vmin=0, vmax=1)
                axes[0, col].set_title(f't={t}', fontsize=9)
                axes[0, col].axis('off')
            
                im = axes[1, col].imshow(hr[1, t], cmap='RdBu_r', vmin=0, vmax=0.5)
                axes[1, col].axis('off')

        axes[0, 0].set_ylabel('U', fontsize=10)
        axes[1, 0].set_ylabel('V', fontsize=10)
        plt.colorbar(im, ax=axes[:, -1], shrink=0.8)
        plt.tight_layout()
        plt.savefig(data_dir / f'vis_IC{ic_number}_{ri_tag}_HR_snapshots.png', dpi=150)
        plt.show()
        print(f"  ✓ Saved HR snapshots")

        # ── 图2：LR vs HR 对比（同一时刻）──────────────────────────
        # LR t=200 对应 HR t=200*4=800
        t_lr_list = [50, 100, 150, 200, 250, 300]
        fig, axes = plt.subplots(4, len(t_lr_list), figsize=(18, 10))
        fig.suptitle(f'IC={ic_number} | LR(32×32) vs HR(256×256) | ri={random_influence}', fontsize=13)

        for col, t_lr in enumerate(t_lr_list):
            t_hr = t_lr * 4  # 时间对齐
        
            # LR U
            axes[0, col].imshow(lr[0, t_lr], cmap='RdBu_r', vmin=0, vmax=1)
            axes[0, col].set_title(f'LR t={t_lr}', fontsize=8)
            axes[0, col].axis('off')
        
            # HR U
            axes[1, col].imshow(hr[0, t_hr], cmap='RdBu_r', vmin=0, vmax=1)
            axes[1, col].set_title(f'HR t={t_hr}', fontsize=8)
            axes[1, col].axis('off')
        
            # LR V
            axes[2, col].imshow(lr[1, t_lr], cmap='RdBu_r', vmin=0, vmax=0.5)
            axes[2, col].axis('off')
        
            # HR V
            axes[3, col].imshow(hr[1, t_hr], cmap='RdBu_r', vmin=0, vmax=0.5)
            axes[3, col].axis('off')

        for ax, label in zip(axes[:, 0], ['LR U', 'HR U', 'LR V', 'HR V']):
            ax.set_ylabel(label, fontsize=10)

        plt.tight_layout()
     #   plt.savefig(data_dir / f'vis_IC{ic_number}_{ri_tag}_LR_vs_HR.png', dpi=150)
        plt.show()
    #    print(f"  ✓ Saved LR vs HR comparison")

        # ── 图3：所有IC的初始状态对比 ──────────────────────────────
    def visualize_all_ICs(data_dir, n_ICs=10, random_influence=0.1, t=0):
        """展示所有IC在同一时刻的U通道，确认IC之间确实不同"""
        ri_tag = f"ri{int(random_influence * 100):02d}"
    
        fig, axes = plt.subplots(2, n_ICs, figsize=(22, 5))
        fig.suptitle(f'All ICs at t={t} | U channel | ri={random_influence}', fontsize=13)

        for ic in range(1, n_ICs + 1):
            hr_path = data_dir / f'2DGS_IC{ic}_{ri_tag}_2x3001x256x256.mat'
            hr = scio.loadmat(hr_path)['uv']
        
            axes[0, ic-1].imshow(hr[0, t], cmap='RdBu_r', vmin=0, vmax=1)
            axes[0, ic-1].set_title(f'IC={ic}', fontsize=9)
            axes[0, ic-1].axis('off')
        
            # 也看一个晚期时间步，确认pattern不同
            axes[1, ic-1].imshow(hr[0, 1500], cmap='RdBu_r', vmin=0, vmax=1)
            axes[1, ic-1].axis('off')

        axes[0, 0].set_ylabel('t=0', fontsize=10)
        axes[1, 0].set_ylabel('t=1500', fontsize=10)
        plt.tight_layout()
       # plt.savefig(data_dir / f'vis_allICs_{ri_tag}_comparison.png', dpi=150)
     #   plt.show()
       # print(f"✓ Saved all-IC comparison")


    # ── 运行 ──────────────────────────────────────────────────────
    if __name__ == '__main__':
        # 单个IC详细可视化
        visualize_dataset(data_dir, ic_number=1, random_influence=0.1)
    
        # 所有IC对比（确认多样性）
        visualize_all_ICs(data_dir, n_ICs=10, random_influence=0.1)


## Optional dataset detail plots

Requires the paired dataset files from the generation cell. Enable the flag to run.


In [ ]:
RUN_DATA_DETAIL_PLOTS = False
if RUN_DATA_DETAIL_PLOTS:
    import scipy.io as scio
    import matplotlib.pyplot as plt
    import numpy as np
    from pathlib import Path

    data_dir = DATA_ROOT / 'gray_scott'
    fig_dir = RESULTS_ROOT / 'data_generation' / 'gray_scott' / 'visualization'
    fig_dir.mkdir(parents=True, exist_ok=True)

    RANDOM_INFLUENCE = 0.1
    ri_tag = f"ri{int(RANDOM_INFLUENCE * 100):02d}"  # → "ri10"

    # ─────────────────────────────────────────────
    # 1. 单文件可视化（单个IC，单个时间步）
    # ─────────────────────────────────────────────
    def visualize_single(ic_number, time_step=0, res='high',
                         random_influence=RANDOM_INFLUENCE, save=False):
        """
        Args:
            ic_number:        IC编号 (1~10)
            time_step:        时间步索引
            res:              'high' (HR 3001步) 或 'low' (LR 751步)
            random_influence: 与生成时一致
            save:             是否保存图片
        """
        ri_tag = f"ri{int(random_influence * 100):02d}"

        if res == 'high':
            fname = f'2DGS_IC{ic_number}_{ri_tag}_2x3001x256x256.mat'
        else:
            fname = f'2DGS_IC{ic_number}_{ri_tag}_2x751x32x32.mat'

        file_path = data_dir / fname
        if not file_path.exists():
            print(f"❌ File not found: {file_path}")
            return

        uv = scio.loadmat(file_path)['uv']  # [2, T, H, W]
        T  = uv.shape[1]

        if time_step >= T:
            print(f"❌ time_step={time_step} out of range (max={T-1})")
            return

        u = uv[0, time_step, :, :]
        v = uv[1, time_step, :, :]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        fig.suptitle(f'IC={ic_number} | {res.upper()} | t={time_step} | {ri_tag}',
                     fontsize=13, fontweight='bold')

        im1 = ax1.imshow(u, cmap='hot', origin='lower')
        ax1.set_title('U (activator)')
        fig.colorbar(im1, ax=ax1)

        im2 = ax2.imshow(v, cmap='viridis', origin='lower')
        ax2.set_title('V (inhibitor)')
        fig.colorbar(im2, ax=ax2)

        plt.tight_layout()

        if save:
            out = fig_dir / f'IC{ic_number}_{ri_tag}_{res}_t{time_step}.png'
            plt.savefig(out, dpi=150, bbox_inches='tight')
            print(f"  ✓ Saved: {out.name}")

        plt.show()


    # ─────────────────────────────────────────────
    # 2. 多IC横向对比（同一时间步，不同IC并排）
    # ─────────────────────────────────────────────
    def visualize_multi_ic(ic_list, time_step=500, res='high',
                           component='u', random_influence=RANDOM_INFLUENCE,
                           save=False):
        """
        Args:
            ic_list:    要对比的IC编号列表，如 [1, 3, 5, 7]
            component:  'u' 或 'v'
            save:       是否保存图片
        """
        ri_tag  = f"ri{int(random_influence * 100):02d}"
        n       = len(ic_list)
        cmap    = 'hot' if component == 'u' else 'viridis'
        ch_idx  = 0    if component == 'u' else 1

        fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
        if n == 1:
            axes = [axes]

        fig.suptitle(f'{component.upper()} component | {res.upper()} | t={time_step} | {ri_tag}',
                     fontsize=13, fontweight='bold')

        for ax, ic in zip(axes, ic_list):
            if res == 'high':
                fname = f'2DGS_IC{ic}_{ri_tag}_2x3001x256x256.mat'
            else:
                fname = f'2DGS_IC{ic}_{ri_tag}_2x751x32x32.mat'

            fpath = data_dir / fname
            if not fpath.exists():
                ax.set_title(f'IC{ic}\n❌ not found')
                ax.axis('off')
                continue

            uv   = scio.loadmat(fpath)['uv']
            field = uv[ch_idx, time_step, :, :]

            im = ax.imshow(field, cmap=cmap, origin='lower')
            ax.set_title(f'IC {ic}')
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        plt.tight_layout()

        if save:
            ic_str = '_'.join(str(i) for i in ic_list)
            out = fig_dir / f'multiIC_{ic_str}_{ri_tag}_{res}_{component}_t{time_step}.png'
            plt.savefig(out, dpi=150, bbox_inches='tight')
            print(f"  ✓ Saved: {out.name}")

        plt.show()


    # ─────────────────────────────────────────────
    # 3. 单个IC的时间演化（多时间步纵向展示）
    # ─────────────────────────────────────────────
    def visualize_time_evolution(ic_number, time_steps, res='high',
                                 component='u', random_influence=RANDOM_INFLUENCE,
                                 save=False):
        """
        Args:
            time_steps: 时间步列表，如 [0, 500, 1000, 1500, 2000, 2500]
        """
        ri_tag  = f"ri{int(random_influence * 100):02d}"
        ch_idx  = 0 if component == 'u' else 1
        cmap    = 'hot' if component == 'u' else 'viridis'

        if res == 'high':
            fname = f'2DGS_IC{ic_number}_{ri_tag}_2x3001x256x256.mat'
        else:
            fname = f'2DGS_IC{ic_number}_{ri_tag}_2x751x32x32.mat'

        fpath = data_dir / fname
        if not fpath.exists():
            print(f"❌ File not found: {fpath}")
            return

        uv = scio.loadmat(fpath)['uv']
        T  = uv.shape[1]
        n  = len(time_steps)

        fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
        if n == 1:
            axes = [axes]

        fig.suptitle(f'IC={ic_number} | {component.upper()} | {res.upper()} | {ri_tag}',
                     fontsize=13, fontweight='bold')

        for ax, t in zip(axes, time_steps):
            if t >= T:
                ax.set_title(f't={t}\n❌ out of range')
                ax.axis('off')
                continue
            im = ax.imshow(uv[ch_idx, t, :, :], cmap=cmap, origin='lower')
            ax.set_title(f't = {t}')
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        plt.tight_layout()

        if save:
            t_str = '_'.join(str(t) for t in time_steps)
            out = fig_dir / f'IC{ic_number}_{ri_tag}_{res}_{component}_evo_{t_str}.png'
            plt.savefig(out, dpi=150, bbox_inches='tight')
            print(f"  ✓ Saved: {out.name}")

        plt.show()


    # ─────────────────────────────────────────────
    # 4. HR vs LR 对比（同一IC同一时间步）
    # ─────────────────────────────────────────────
    def visualize_hr_vs_lr(ic_number, hr_t=2500, lr_t=625,
                           random_influence=RANDOM_INFLUENCE, save=False):
        """
        Args:
            hr_t: HR时间步索引（0~3000）
            lr_t: 对应的LR时间步索引（hr_t // 4，0~750）
        """
        ri_tag = f"ri{int(random_influence * 100):02d}"

        hr_path = data_dir / f'2DGS_IC{ic_number}_{ri_tag}_2x3001x256x256.mat'
        lr_path = data_dir / f'2DGS_IC{ic_number}_{ri_tag}_2x751x32x32.mat'

        if not hr_path.exists() or not lr_path.exists():
            print(f"❌ Files not found for IC{ic_number}")
            return

        uv_hr = scio.loadmat(hr_path)['uv']
        uv_lr = scio.loadmat(lr_path)['uv']

        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        fig.suptitle(f'HR vs LR | IC={ic_number} | {ri_tag}', fontsize=13, fontweight='bold')

        pairs = [
            (axes[0, 0], uv_hr[0, hr_t], f'HR U  (t={hr_t})', 'hot'),
            (axes[0, 1], uv_lr[0, lr_t], f'LR U  (t={lr_t})', 'hot'),
            (axes[1, 0], uv_hr[1, hr_t], f'HR V  (t={hr_t})', 'viridis'),
            (axes[1, 1], uv_lr[1, lr_t], f'LR V  (t={lr_t})', 'viridis'),
        ]

        for ax, field, title, cmap in pairs:
            im = ax.imshow(field, cmap=cmap, origin='lower')
            ax.set_title(title)
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        plt.tight_layout()

        if save:
            out = fig_dir / f'IC{ic_number}_{ri_tag}_hr_vs_lr.png'
            plt.savefig(out, dpi=150, bbox_inches='tight')
            print(f"  ✓ Saved: {out.name}")

        plt.show()


    # ─────────────────────────────────────────────
    # 示例调用
    # ─────────────────────────────────────────────
    if __name__ == '__main__':

        # 1. 查看单个IC的某一时间步
        visualize_single(ic_number=1, time_step=2500, res='high', save=False)

        # 2. 对比4个IC在同一时间步的差异（验证IC多样性）
        visualize_multi_ic(ic_list=[1, 3, 5, 7], time_step=500, res='high',
                           component='u', save=False)

        # 3. 查看IC1的时间演化过程
        visualize_time_evolution(ic_number=1,
                                 time_steps=[0, 500, 1000, 1500, 2000, 2500],
                                 res='high', component='u', save=False)

        # 4. HR vs LR 分辨率对比
        visualize_hr_vs_lr(ic_number=1, hr_t=2500, lr_t=625, save=False)
